## Install & Import Libraries

In [1]:
!pip install transformers datasets scikit-learn

import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

C:\Users\Samyagh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

In [2]:
!python -m pip install --upgrade pip

In [3]:
!pip install --upgrade transformers

In [4]:
df = pd.read_csv(r"C:\Users\Samyagh\Desktop\IMDB Dataset.csv")
df = df.sample(1000, random_state=42)
df.head()

,review,sentiment
33553,I really liked this Summerslam due to the look...,positive
9427,Not many television shows appeal to quite as m...,positive
199,The film quickly gets to a major chase scene w...,negative
12447,Jane Austen would definitely approve of this o...,positive
39489,Expectations were somewhat high for me when I ...,negative


In [5]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

## Preprocessing

In [6]:
def clean_text(text):
    text = text.lower()
    text = text.replace("<br />", "")
    return text

df['review'] = df['review'].apply(clean_text)

# Check nulls
df = df.dropna()

## Train / Validation / Test Split

In [7]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42)

## Tokenization (BERT)

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

## Create Dataset Class

In [9]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)
test_dataset = Dataset(test_encodings, test_labels)

## Load BERT Model

In [10]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3981.75it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initializ

## Training Setup

In [11]:
!pip install accelerate

In [12]:
import sys
!{sys.executable} -m pip install --upgrade transformers accelerate torch

In [13]:
import accelerate
print(accelerate.__version__)

1.13.0


In [14]:
from transformers import TrainingArguments
import os

os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,             
    eval_strategy="epoch",
    save_strategy="epoch"
)

In [15]:
import transformers
print(transformers.__version__)

5.5.0


## Metrics Function

In [16]:
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

## Trainer

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

## Train Model

In [18]:
trainer.train()

C:\Users\Samyagh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.481719,0.780000,0.744186,0.744186,0.744186


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.17it/s]


TrainOutput(global_step=200, training_loss=0.5231918716430664, metrics={'train_runtime': 439.4752, 'train_samples_per_second': 1.82, 'train_steps_per_second': 0.455, 'total_flos': 52622211072000.0, 'train_loss': 0.5231918716430664, 'epoch': 1.0})

## Evaluate Model

In [19]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

C:\Users\Samyagh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [20]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
accuracy = accuracy_score(labels, preds)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.82
Precision: 0.8048780487804879
Recall: 0.7674418604651163
F1 Score: 0.7857142857142857


## Confusion Matrix

In [21]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

cm = confusion_matrix(test_labels, preds)
print(cm)

C:\Users\Samyagh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[[49  8]
 [10 33]]


## Experiment 1: Freeze BERT

In [25]:
for param in model.bert.parameters():
    param.requires_grad = False

## Experiment 2: Fine-tune last 2 layers

In [22]:
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False